# Dependency Test Notebook

Minimal test to validate Phase 1 dependencies before running full collection.

**What this does:**
- Tests specific dependency versions
- Tests basic imports
- Attempts model load (without forward pass)
- Reports which versions work
- Then you update requirements.txt and re-run to confirm

**Runtime:** ~5-10 minutes

In [ ]:
# Clone repo (use integration branch with Phase 1 code)
!rm -rf /content/moe-expert-prefetching
!cd /content && git clone --branch integration/phase-1-validation https://github.com/scerruti/moe-expert-prefetching.git
%cd /content/moe-expert-prefetching/phase_1/scripts

In [ ]:
# Test specific versions (not from requirements.txt yet)
# This lets us find what works, then update requirements.txt
print("Testing versions:")
print("  torch==2.11.0")
print("  torchvision==0.16.0")
print("  transformers==4.40.2")
print("  datasets==5.0.1")
print("  pyarrow==25.0.1\n")

!pip install -q torch==2.11.0 torchvision==0.16.0 transformers==4.40.2 datasets==5.0.1 pyarrow==25.0.1 tqdm accelerate bitsandbytes
print("\n✅ Test versions installed")

In [ ]:
# Test 1: Basic imports
print("Test 1: Basic imports...")
try:
    import torch
    import transformers
    import datasets
    import pyarrow
    print(f"✅ torch {torch.__version__}")
    print(f"✅ transformers {transformers.__version__}")
    print(f"✅ datasets {datasets.__version__}")
    print(f"✅ pyarrow {pyarrow.__version__}")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    raise

In [ ]:
# Test 2: Module imports
print("\nTest 2: Phase 1 module imports...")
try:
    import sys
    sys.path.insert(0, '/content/moe-expert-prefetching/phase_1/scripts')
    
    from model_loader import ModelLoader
    from dataset_loaders import GSM8KLoader, MBPPLoader
    from tokenizer import TokenizationPipeline
    print("✅ ModelLoader imported")
    print("✅ Dataset loaders imported")
    print("✅ TokenizationPipeline imported")
except ImportError as e:
    print(f"❌ Module import failed: {e}")
    raise

In [ ]:
# Test 3: Model loading
print("\nTest 3: Model loading (this takes ~2 min)...")
try:
    loader = ModelLoader("qwen1.5-moe-a2.7b")
    print(f"  Loading {loader.model_id}...")
    model, tokenizer, config = loader.load()
    print(f"✅ Model loaded successfully")
    print(f"  Device: {loader.device}")
    print(f"  Dtype: {loader.dtype}")
    print(f"  Experts: {config['num_experts']}, Top-K: {config['top_k']}")
except Exception as e:
    print(f"❌ Model loading failed: {e}")
    raise

In [ ]:
# Test 4: GPU verification
print("\nTest 4: GPU availability...")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("✅ GPU ready for Phase 1 collection")
else:
    print("⚠️  No GPU available (Phase 1 will run very slowly)")

In [ ]:
# Summary
print("\n" + "="*60)
print("✅ ALL DEPENDENCY TESTS PASSED")
print("="*60)
print("\nThese versions work:")
print("  torch==2.11.0")
print("  torchvision==0.16.0")
print("  transformers==4.40.2")
print("  datasets==5.0.1")
print("  pyarrow==25.0.1")
print("\nNext steps:")
print("1. If you haven't already, update requirements.txt with these versions")
print("2. Re-run this notebook once to confirm (it will install from requirements.txt)")
print("3. Then run Phase1_Full_Collection_Colab.ipynb for Phase 1")